# Acetic Acid

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
acetic,60.052,2.77367,2.967108595,184.2518865,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
acetic,H,acetic,e,2000.0,0.03
"""

model = PCSAFT(["acetic"], userlocations = [like_parameter, assoc_parameter])

PCSAFT{BasicIdeal, Float64} with 1 component:
 "acetic"
Contains parameters: Mw, segment, sigma, epsilon, epsilon_assoc, bondvol

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_acetic.csv")
fix_line_endings("rhol_acetic.csv")

Fixed: satp_acetic.csv
Fixed: rhol_acetic.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

my_liquid_density (generic function with 1 method)

In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 500.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 0.50,
        :guess   => 0.4
    ),
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 500.0)
 Dict(:upper => 0.5, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.001)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_acetic.csv",
        "satp_acetic.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.006250359386127295


In [7]:
method = ECA(; options = Options(iterations = 10000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2375.020390746329, 0.4139486409186755], PCSAFT{BasicIdeal, Float64}("acetic"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_acetic.csv",   my_saturation_p)


=== AAD: satp_acetic.csv ===

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



Clapeyron Estimator  exp           calc          ARD%    
273.1500    466.620000    432.008600    7.4175  
283.1500    845.248800    834.007659    1.3299  
293.1500    1573.176000   1532.825254   2.5649  
303.1500    2653.068000   2695.924926   1.6154  
313.1500    4532.880000   4557.907140   0.5521  
323.1500    7492.584000   7436.432290   0.7494  
333.1500    11772.156000  11748.740393  0.1989  
343.1500    18278.172000  18028.108820  1.3681  
353.1500    26943.972000  26939.571765  0.0163  
363.1500    39036.096000  39294.254462  0.6613  
373.1500    55527.780000  56061.745359  0.9616  
383.1500    77672.232000  78380.032607  0.9113  
403.1500    138652.800000  145102.841216  4.6519  
413.1500    184114.920000  192674.612578  4.6491  
423.1500    249975.000000  252130.814090  0.8624  
433.1500    321167.880000  325498.335908  1.3483  
443.1500    407692.560000  414970.730169  1.7852  
453.1500    511015.560000  522898.569710  2.3254  
463.1500    631003.560000  651777.912640  3.292

1.9622744527576395

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_acetic.csv",   my_liquid_density)


=== AAD: rhol_acetic.csv ===
Clapeyron Estimator  exp           calc          ARD%    
293.1500    1049.100000   1063.918718   1.4125  
303.1500    1039.200000   1052.521377   1.2819  
313.1500    1028.400000   1041.034753   1.2286  
323.1500    1017.500000   1029.425443   1.1720  
333.1500    1006.000000   1017.661004   1.1591  
343.1500    994.800000    1005.709581   1.0967  
353.1500    983.500000    993.539556    1.0208  
363.1500    971.800000    981.119193    0.9590  
373.1500    959.900000    968.416284    0.8872  
383.1500    948.300000    955.397782    0.7485  
393.1500    936.200000    942.029405    0.6227  
403.1500    923.500000    928.275217    0.5171  
413.1500    909.100000    914.097142    0.5497  
423.1500    896.300000    899.454428    0.3519  
433.1500    882.900000    884.303005    0.1589  
443.1500    869.400000    868.594726    0.0926  
453.1500    855.500000    852.276444    0.3768  
463.1500    841.300000    835.288862    0.7145  
473.1500    826.500000    817.

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


2.83321090221719